# Inciting speech — LLooM workbench viewer (all 6 runs)

Loads LLooM outputs from disk for **one** of the six pipelines in `LLooM_Inciting_FN_FP.ipynb` (three classifiers × FN/FP) and shows the interactive **`l.vis()`** widget. No pipeline / API calls.

**Data source** (set `_LLOOM_REL` below): directory under `outputs/revised_lloom/{classifier}/{fn|fp}/` containing either
- **`*.pkl`** — newest file is used if `LLOOM_SESSION_PKL` is `None`, or
- **`lloom_scores_long.csv`** — reconstructs session if no pickle (optional **`lloom_distilled_bullets.csv`**).

Run from the **`error_analysis`** directory (or adjust paths). Requires Jupyter widget support for `text_lloom`.

In [ ]:
!pip install -q -U text_lloom nest_asyncio ipywidgets pandas

In [ ]:
import glob
import os
import pickle

import nest_asyncio
import pandas as pd
import text_lloom.workbench as wb
from text_lloom.concept import Concept
from text_lloom.llm import OpenAIModel, OpenAIEmbedModel

nest_asyncio.apply()

In [ ]:
# --- Configure: one folder under outputs/revised_lloom/<classifier>/<fn|fp>/ from LLooM_Inciting_FN_FP.ipynb ---
_BASE = os.path.join("outputs", "revised_lloom")

# Pick exactly one active path (uncomment another line and comment the current one to switch the 6 categories):
_LLOOM_REL = os.path.join(_BASE, "identity", "fn")
# _LLOOM_REL = os.path.join(_BASE, "identity", "fp")
# _LLOOM_REL = os.path.join(_BASE, "imputed_misdeeds", "fn")
# _LLOOM_REL = os.path.join(_BASE, "imputed_misdeeds", "fp")
# _LLOOM_REL = os.path.join(_BASE, "exhortation", "fn")
# _LLOOM_REL = os.path.join(_BASE, "exhortation", "fp")

LLOOM_SESSION_PKL = None  # e.g. "lloom_session.pkl"; None = newest *.pkl in folder


def _resolve_lloom_folder(preferred: str) -> str:
    """Resolve path whether cwd is error_analysis, Inciting/error_analysis, or repo root."""
    cands = [
        preferred,
        os.path.join("error_analysis", preferred),
        os.path.join("Inciting", "error_analysis", preferred),
    ]
    for c in cands:
        ap = os.path.abspath(c)
        if os.path.isdir(ap):
            return ap
    return os.path.abspath(preferred)


LLOOM_VIS_FOLDER = _resolve_lloom_folder(_LLOOM_REL)

_SCORE_COLS = [
    "doc_id",
    "text",
    "concept_id",
    "concept_name",
    "concept_prompt",
    "score",
    "rationale",
    "highlight",
    "concept_seed",
]


def _norm_doc_id(v):
    if pd.isna(v):
        return ""
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    s = str(v).strip()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except ValueError:
        pass
    return s


def _ensure_vis_auxiliary_dfs(l):
    """prep_vis_dfs expects df_filtered / df_bullets with exactly one text column besides id."""
    id_col, txt_col = l.doc_id_col, l.doc_col
    l.in_df = l.in_df.copy()
    l.in_df[id_col] = l.in_df[id_col].map(_norm_doc_id)
    for cid, sdf in list(getattr(l, "results", {}).items()):
        sdf = sdf.copy()
        if id_col in sdf.columns:
            sdf[id_col] = sdf[id_col].map(_norm_doc_id)
        if "doc_id" in sdf.columns and id_col != "doc_id":
            sdf["doc_id"] = sdf["doc_id"].map(_norm_doc_id)
        if "concept_id" in sdf.columns:
            sdf["concept_id"] = sdf["concept_id"].astype(str)
        l.results[cid] = sdf
    base = l.in_df[[id_col, txt_col]].copy()
    l.df_filtered = base.copy()
    bb = getattr(l, "df_bullets", None)
    if bb is None or len(bb) == 0:
        l.df_bullets = pd.DataFrame({id_col: base[id_col].values, "text": ""})
    else:
        bb = bb.copy()
        bb[id_col] = bb[id_col].map(_norm_doc_id)
        non_id = [c for c in bb.columns if c != id_col]
        if len(non_id) == 1:
            l.df_bullets = bb[[id_col, non_id[0]]].rename(columns={non_id[0]: "text"})
        elif "text" in bb.columns:
            l.df_bullets = bb[[id_col, "text"]].copy()
        else:
            l.df_bullets = pd.DataFrame({id_col: base[id_col].values, "text": ""})


def load_lloom_for_vis(folder: str, pkl_basename: str | None = None):
    folder = os.path.abspath(folder)
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Not a directory: {folder}")

    pkl_path = None
    if pkl_basename:
        cand = os.path.join(folder, pkl_basename)
        if os.path.isfile(cand):
            pkl_path = cand
    if pkl_path is None:
        pkls = sorted(glob.glob(os.path.join(folder, "*.pkl")), key=os.path.getmtime, reverse=True)
        if pkls:
            pkl_path = pkls[0]

    if pkl_path:
        print(f"Loading pickled session: {pkl_path}")
        with open(pkl_path, "rb") as f:
            l = pickle.load(f)
        l.select_widget = None
        for c_id, c in l.concepts.items():
            c.active = c_id in getattr(l, "results", {})
        _ensure_vis_auxiliary_dfs(l)
        return l

    scores_csv = os.path.join(folder, "lloom_scores_long.csv")
    if not os.path.isfile(scores_csv):
        raise FileNotFoundError(
            f"No .pkl in {folder} and missing {scores_csv}. Run LLooM_Inciting_FN_FP.ipynb or save a session."
        )

    print(f"Reconstructing session from {scores_csv} (no .pkl found)")
    long_df = pd.read_csv(scores_csv)
    for col in _SCORE_COLS:
        if col not in long_df.columns:
            raise ValueError(f"CSV missing column {col!r}; got {list(long_df.columns)}")

    long_df = long_df.copy()
    long_df["doc_id"] = long_df["doc_id"].map(_norm_doc_id)
    in_df = (
        long_df.sort_values("doc_id")
        .drop_duplicates(subset=["doc_id"], keep="first")[["doc_id", "text"]]
        .reset_index(drop=True)
    )

    concepts = {}
    results = {}
    for cid, g in long_df.groupby("concept_id"):
        cid = str(cid).strip()
        row0 = g.iloc[0]
        c = Concept(
            name=str(row0["concept_name"]),
            prompt=str(row0["concept_prompt"]),
            example_ids=[],
            active=True,
            summary=None,
            seed=None if pd.isna(row0.get("concept_seed")) else row0.get("concept_seed"),
        )
        c.id = cid
        concepts[cid] = c
        sub = g[_SCORE_COLS].copy()
        sub["doc_id"] = sub["doc_id"].map(_norm_doc_id)
        sub["concept_id"] = sub["concept_id"].astype(str)
        results[cid] = sub.reset_index(drop=True)

    api_key = os.environ.get("OPENAI_API_KEY", "sk-dummy-not-used-for-vis-only")
    l = wb.lloom(
        in_df,
        text_col="text",
        id_col="doc_id",
        distill_model=OpenAIModel(name="gpt-3.5-turbo", api_key=api_key),
        cluster_model=OpenAIEmbedModel(name="text-embedding-3-large", api_key=api_key),
        synth_model=OpenAIModel(name="gpt-3.5-turbo", api_key=api_key),
        score_model=OpenAIModel(name="gpt-3.5-turbo", api_key=api_key),
    )
    l.concepts = concepts
    l.results = results
    l.select_widget = None
    l.df_filtered = in_df.copy()
    bullets_path = os.path.join(folder, "lloom_distilled_bullets.csv")
    if os.path.isfile(bullets_path):
        l.df_bullets = pd.read_csv(bullets_path)
        l.df_bullets["doc_id"] = l.df_bullets["doc_id"].map(_norm_doc_id)
    else:
        l.df_bullets = None

    _ensure_vis_auxiliary_dfs(l)
    return l


print(f"Using folder: {LLOOM_VIS_FOLDER}")
l = load_lloom_for_vis(LLOOM_VIS_FOLDER, pkl_basename=LLOOM_SESSION_PKL)

_seen = {}
for _cid, _c in l.concepts.items():
    _n = _c.name
    _seen[_n] = _seen.get(_n, 0) + 1
    if _seen[_n] > 1:
        _new = f"{_n} ({_seen[_n]})"
        _c.name = _new
        if _cid in l.results:
            _sdf = l.results[_cid].copy()
            _sdf["concept_name"] = _new
            l.results[_cid] = _sdf

l.vis()
